In [ ]:
# ─────────────────────────────────────────────
# CELL 1 — Read Bronze CSV files into DataFrames
# Using inferSchema for automatic type detection
# ─────────────────────────────────────────────

# Bronze file paths in Lakehouse
BRONZE_PATH = "Files/bronze"

# Read all 4 bronze CSV files
df_employees = spark.read.csv(
    f"{BRONZE_PATH}/employees/employees.csv",
    inferSchema=True,
    header=True
)

df_performance = spark.read.csv(
    f"{BRONZE_PATH}/performance/performance.csv",
    inferSchema=True,
    header=True
)

df_satisfaction = spark.read.csv(
    f"{BRONZE_PATH}/satisfaction/satisfaction.csv",
    inferSchema=True,
    header=True
)

df_attendance = spark.read.csv(
    f"{BRONZE_PATH}/attendance/attendance.csv",
    inferSchema=True,
    header=True
)

print("✅ All Bronze files loaded successfully")
print(f"   Employees    : {df_employees.count()} rows, {len(df_employees.columns)} cols")
print(f"   Performance  : {df_performance.count()} rows, {len(df_performance.columns)} cols")
print(f"   Satisfaction : {df_satisfaction.count()} rows, {len(df_satisfaction.columns)} cols")
print(f"   Attendance   : {df_attendance.count()} rows, {len(df_attendance.columns)} cols")

In [14]:
# ─────────────────────────────────────────────
# CELL 2 — Schema Validation + Null Checks
# Verify data types and data quality
# before any transformation
# ─────────────────────────────────────────────

print("=" * 50)
print("EMPLOYEES SCHEMA")
print("=" * 50)
df_employees.printSchema()

print("=" * 50)
print("NULL COUNTS — EMPLOYEES")
print("=" * 50)
from pyspark.sql.functions import col, sum as spark_sum

# Count nulls in every column of employees table
null_counts = df_employees.select([
    spark_sum(
        col(c).isNull().cast("int")
    ).alias(c)
    for c in df_employees.columns
])
null_counts.show()

print("=" * 50)
print("SAMPLE DATA — EMPLOYEES")
print("=" * 50)
df_employees.show(5)

StatementMeta(, 5e1a6aa7-da8b-43a3-b2b2-8dd481f494c7, 38, Finished, Available, Finished, False)

EMPLOYEES SCHEMA
root
 |-- EmployeeID: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- MaritalStatus: string (nullable = true)
 |-- Education: integer (nullable = true)
 |-- EducationField: string (nullable = true)
 |-- Department: string (nullable = true)
 |-- JobRole: string (nullable = true)
 |-- JobLevel: integer (nullable = true)
 |-- MonthlyIncome: integer (nullable = true)
 |-- YearsAtCompany: integer (nullable = true)
 |-- NumCompaniesWorked: integer (nullable = true)
 |-- DistanceFromHome: integer (nullable = true)
 |-- Attrition: string (nullable = true)

NULL COUNTS — EMPLOYEES
+----------+----+---+------+-------------+---------+--------------+----------+-------+--------+-------------+--------------+------------------+----------------+---------+
|EmployeeID|Name|Age|Gender|MaritalStatus|Education|EducationField|Department|JobRole|JobLevel|MonthlyIncome|YearsAtCompany|NumCompaniesWork

In [15]:
# ─────────────────────────────────────────────
# CELL 3 — Register temp views for Spark SQL
# ─────────────────────────────────────────────

df_employees.createOrReplaceTempView("employees")
df_performance.createOrReplaceTempView("performance")
df_satisfaction.createOrReplaceTempView("satisfaction")
df_attendance.createOrReplaceTempView("attendance")

print("✅ All temp views registered")

StatementMeta(, 5e1a6aa7-da8b-43a3-b2b2-8dd481f494c7, 39, Finished, Available, Finished, True)

✅ All temp views registered


In [16]:
%%sql
-- ─────────────────────────────────────────────
-- SQL Warmups
-- ─────────────────────────────────────────────
SELECT * FROM employees LIMIT 1;
SELECT * FROM performance LIMIT 1;
SELECT * FROM satisfaction LIMIT 1;
SELECT * FROM attendance LIMIT 1;

-- Show all columns from employees table for the first 5 rows.
SELECT * FROM employees LIMIT 5;

-- Count how many employees are in each Department. Order by count descending.
SELECT 
    Department,
    COUNT(EmployeeID) AS TotalEmployees
FROM employees
GROUP BY Department
ORDER BY TotalEmployees DESC;

-- Calculate attrition rate percentage per Department.
-- Show Department, TotalEmployees, AttritionCount, AttritionRate.
-- Round AttritionRate to 2 decimal places.
SELECT 
    Department,
    COUNT(EmployeeID) AS TotalEmployees,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS AttritionCount,
    ROUND(
        SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0 
        / COUNT(EmployeeID)
    , 2) AS AttritionRate
FROM employees
GROUP BY Department
ORDER BY AttritionRate DESC;


-- Join employees and performance tables on EmployeeID.
-- Show EmployeeID, Name, Department, OverTime, PerformanceRating
-- Only for employees where Attrition = 'Yes'
SELECT 
    e.EmployeeID, 
    e.Name, e.Department, 
    p.OverTime, 
    p.PerformanceRating 
FROM employees e 
INNER JOIN performance p 
ON e.EmployeeID = p.EmployeeID
WHERE e.Attrition = 'Yes';


-- Add a new column called RiskLevel based on these rules:
-- JobSatisfaction = 1 OR WorkLifeBalance = 1 → 'High Risk'
-- JobSatisfaction = 2 OR WorkLifeBalance = 2 → 'Medium Risk'
-- Everything else → 'Low Risk'
-- Show EmployeeID, JobSatisfaction, WorkLifeBalance, RiskLevel
SELECT 
    EmployeeID, JobSatisfaction, WorkLifeBalance,
    CASE 
        WHEN JobSatisfaction = 1 OR WorkLifeBalance = 1 THEN 'High Risk'
        WHEN JobSatisfaction = 2 OR WorkLifeBalance = 2 THEN 'Medium Risk'
        ELSE 'Low Risk'
    END AS RiskLevel
FROM satisfaction;


StatementMeta(, 5e1a6aa7-da8b-43a3-b2b2-8dd481f494c7, 48, Finished, Available, Finished, True)

<Spark SQL result set with 1 rows and 15 fields>

<Spark SQL result set with 1 rows and 7 fields>

<Spark SQL result set with 1 rows and 7 fields>

<Spark SQL result set with 1 rows and 6 fields>

<Spark SQL result set with 5 rows and 15 fields>

<Spark SQL result set with 10 rows and 2 fields>

<Spark SQL result set with 10 rows and 4 fields>

<Spark SQL result set with 1000 rows and 5 fields>

<Spark SQL result set with 1000 rows and 4 fields>

In [22]:
%%sql
-- ─────────────────────────────────────────────
-- The Silver Master Query
-- Silver transformation does 4 things:
-- 1. JOIN all 4 tables into one master employee table
-- 2. Clean nulls
-- 3. Add derived columns (RiskLevel, AttritionFlag)
-- 4. Save as Delta table
-- ─────────────────────────────────────────────

SELECT 
    -- From employees table
    e.EmployeeID,	
    e.Name,	
    e.Age,	
    e.Gender,	
    e.MaritalStatus,	
    e.Education,	
    e.EducationField,	
    e.Department,	
    e.JobRole,	
    e.JobLevel,
    e.MonthlyIncome,	
    e.YearsAtCompany,
    e.NumCompaniesWorked,
    e.DistanceFromHome,
    e.Attrition,
    
    -- From performance table 
    p.OverTime, 
    p.PerformanceRating, 
    p.YearsSinceLastPromotion,
    p.TrainingTimesLastYear, 
    p.YearsInCurrentRole, 
    p.TotalWorkingYears,

    -- From satisfaction table
    s.JobSatisfaction,
    s.WorkLifeBalance,
    s.RelationshipSatisfaction,
    s.EnvironmentSatisfaction,
    s.JobInvolvement,
    s.StockOptionLevel,

    -- From attendance table
    a.AvgHoursPerDay, 
    a.AvgLaptopActiveHours,
    a.LeaveDaysTaken, 
    a.AbsentDaysLastYear, 
    a.BusinessTravelFrequency,

    -- Derived columns
    -- 1. AttritionFlag → Attrition Yes=1, No=0
    CASE 
        WHEN e.Attrition = 'Yes' THEN 1
        ELSE 0
    END AS AttritionFlag,

    -- 2. RiskLevel → High/Medium/Low (you already wrote this!)
    CASE 
        WHEN s.JobSatisfaction = 1 OR s.WorkLifeBalance = 1 THEN 'High Risk'
        WHEN s.JobSatisfaction = 2 OR s.WorkLifeBalance = 2 THEN 'Medium Risk'
        ELSE 'Low Risk'
    END AS RiskLevel,

    -- 3. SalaryBand → 'Below 50K', '50K-100K', '100K-200K', 'Above 200K'
    CASE 
        WHEN e.MonthlyIncome < 50000 THEN 'Below 50K'
        WHEN e.MonthlyIncome >= 50000 AND e.MonthlyIncome <= 100000 THEN '50K-100K'
        WHEN e.MonthlyIncome > 100000 AND e.MonthlyIncome <= 200000 THEN '100K-200K'
        WHEN e.MonthlyIncome > 200000 THEN 'Above 200K'
    END AS SalaryBand

FROM employees e 
INNER JOIN performance p 
ON e.EmployeeID = p.EmployeeID
INNER JOIN satisfaction s 
ON e.EmployeeID = s.EmployeeID 
INNER JOIN attendance a 
ON e.EmployeeID = a.EmployeeID ; 

StatementMeta(, 5e1a6aa7-da8b-43a3-b2b2-8dd481f494c7, 63, Finished, Available, Finished, False)

<Spark SQL result set with 1000 rows and 35 fields>

In [20]:
# ─────────────────────────────────────────────
# CELL 4 — Silver Master Transformation
# JOIN all 4 Bronze tables + add derived columns
# ─────────────────────────────────────────────

df_silver_master = spark.sql("""
    SELECT 
        e.EmployeeID, e.Name, e.Age, e.Gender,
        e.MaritalStatus, e.Education, e.EducationField,
        e.Department, e.JobRole, e.JobLevel,
        e.MonthlyIncome, e.YearsAtCompany,
        e.NumCompaniesWorked, e.DistanceFromHome,
        e.Attrition,

        p.OverTime, p.PerformanceRating,
        p.YearsSinceLastPromotion, p.TrainingTimesLastYear,
        p.YearsInCurrentRole, p.TotalWorkingYears,

        s.JobSatisfaction, s.WorkLifeBalance,
        s.RelationshipSatisfaction, s.EnvironmentSatisfaction,
        s.JobInvolvement, s.StockOptionLevel,

        a.AvgHoursPerDay, a.AvgLaptopActiveHours,
        a.LeaveDaysTaken, a.AbsentDaysLastYear,
        a.BusinessTravelFrequency,

        CASE 
            WHEN e.Attrition = 'Yes' THEN 1 
            ELSE 0 
        END AS AttritionFlag,

        CASE 
            WHEN s.JobSatisfaction = 1 OR s.WorkLifeBalance = 1 THEN 'High Risk'
            WHEN s.JobSatisfaction = 2 OR s.WorkLifeBalance = 2 THEN 'Medium Risk'
            ELSE 'Low Risk'
        END AS RiskLevel,

        CASE 
            WHEN e.MonthlyIncome < 50000 THEN 'Below 50K'
            WHEN e.MonthlyIncome >= 50000 
                AND e.MonthlyIncome <= 100000 THEN '50K-100K'
            WHEN e.MonthlyIncome > 100000 
                AND e.MonthlyIncome <= 200000 THEN '100K-200K'
            WHEN e.MonthlyIncome > 200000 THEN 'Above 200K'
        END AS SalaryBand

    FROM employees e 
    INNER JOIN performance p ON e.EmployeeID = p.EmployeeID
    INNER JOIN satisfaction s ON e.EmployeeID = s.EmployeeID 
    INNER JOIN attendance a ON e.EmployeeID = a.EmployeeID
    WHERE e.EmployeeID IS NOT NULL
      AND e.Age IS NOT NULL
      AND e.MonthlyIncome IS NOT NULL
""")

print(f"✅ Silver master table created: {df_silver_master.count()} rows")
print(f"   Total columns: {len(df_silver_master.columns)}")
df_silver_master.show(3)    

StatementMeta(, 5e1a6aa7-da8b-43a3-b2b2-8dd481f494c7, 60, Finished, Available, Finished, False)

✅ Silver master table created: 4000 rows
   Total columns: 35
+----------+----------------+---+------+-------------+---------+--------------+----------------+--------------------+--------+-------------+--------------+------------------+----------------+---------+--------+-----------------+-----------------------+---------------------+------------------+-----------------+---------------+---------------+------------------------+-----------------------+--------------+----------------+--------------+--------------------+--------------+------------------+-----------------------+-------------+-----------+----------+
|EmployeeID|            Name|Age|Gender|MaritalStatus|Education|EducationField|      Department|             JobRole|JobLevel|MonthlyIncome|YearsAtCompany|NumCompaniesWorked|DistanceFromHome|Attrition|OverTime|PerformanceRating|YearsSinceLastPromotion|TrainingTimesLastYear|YearsInCurrentRole|TotalWorkingYears|JobSatisfaction|WorkLifeBalance|RelationshipSatisfaction|EnvironmentSat

In [21]:
# ─────────────────────────────────────────────
# CELL 5 — Save Silver Master as Delta Table
# This lands in Lakehouse Tables section
# Overwrite mode — safe to rerun pipeline
# ─────────────────────────────────────────────

df_silver_master.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_master_employees")

print("✅ Silver Delta table saved successfully")
print("   Table: silver_master_employees")
print("   Location: TechNova_HR_Lakehouse → Tables")
print("   Format: Delta Lake")

StatementMeta(, 5e1a6aa7-da8b-43a3-b2b2-8dd481f494c7, 62, Finished, Available, Finished, False)

✅ Silver Delta table saved successfully
   Table: silver_master_employees
   Location: TechNova_HR_Lakehouse → Tables
   Format: Delta Lake
